## Test Raptor

In [ ]:
'''!pip install -U \
  langchain-chroma \
  langchain-google-genai \
  chromadb'''


In [ ]:
import os
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from raptor_pipeline import (
    RaptorBuilder,
    RaptorRetriever,
)

# ----------------
# Config
# ----------------
COLLECTION_NAME = "medical_papers"
PERSIST_DIR = "./.raptor_checkpoints"

assert os.getenv("GOOGLE_API_KEY"), "❌ GOOGLE_API_KEY not set"

# ----------------
# Load RAPTOR tree (READ ONLY)
# ----------------
builder = RaptorBuilder.from_existing(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

nodes = builder.nodes
tree_structure = builder.tree_structure

print(f"✅ RAPTOR loaded | Nodes: {len(nodes)} | Levels: {len(tree_structure)}")

# ----------------
# Load VectorStore (question → embedding happens here)
# ----------------
embeddings = GoogleGenerativeAIEmbeddings(
    model="text-embedding-005",
    vertexai=True,
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=f"{PERSIST_DIR}/{COLLECTION_NAME}/chroma",
)

# ----------------
# Create Retriever
# ----------------
retriever = RaptorRetriever(
    vectorstore=vectorstore,
    nodes=nodes,
    tree_structure=tree_structure,
    default_mode="tree_traversal",
)

# ----------------
# Question (based on your document)
# ----------------
question = (
    "How is the right ventricular Tei index used to predict postoperative "
    "complications in patients undergoing cardiovascular surgery?"
)

print("\n🧠 QUESTION:")
print(question)

# ----------------
# Retrieve RAPTOR context
# ----------------
context = retriever.get_context(
    query=question,
    top_k=3,
    mode="tree_traversal",
    include_parents=True,
)

print("\n📄 RAPTOR CONTEXT:")
print(context)


✅ RAPTOR loaded | Nodes: 15 | Levels: 3

🧠 QUESTION:
How is the right ventricular Tei index used to predict postoperative complications in patients undergoing cardiovascular surgery?

📄 RAPTOR CONTEXT:
[Level 2] [Source: 41355478_ECG Markers of Positive Drug Challenge With Ajmaline in Patients With Brugada Syndrome.pdf]
Two distinct studies are presented.

The first study identified baseline electrocardiogram (ECG) markers that predict a positive ajmaline challenge (AC) in patients suspected of Brugada Syndrome (BS) but lacking a spontaneous diagnostic ECG. Analyzing 221 patients, researchers found that prominent S-waves in lead II and J-waves in V1 were significant predictors of a positive AC (observed in 42% of patients). Specifically, an S-wave duration in lead II ≥ 19 ms showed 96% sensitivity, and a J-wave amplitude in V1 ≥ 0.05 mV was strongly associated with a positive AC. The study concluded these subtle abnormalities can help identify individuals with a higher likelihood of a 

# Raptor + Agent

In [1]:
import os
from langchain_chroma import Chroma
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)

from raptor_pipeline import (
    RaptorBuilder,
    RaptorRetriever,
)

# Raptor reading:

COLLECTION_NAME = "medical_papers"
PERSIST_DIR = "./.raptor_checkpoints"

assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set"

# Load tree
builder = RaptorBuilder.from_existing(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

nodes = builder.nodes
tree_structure = builder.tree_structure

print(f"✅ RAPTOR loaded | Nodes: {len(nodes)} | Levels: {len(tree_structure)}")

# Vectorstore
embeddings = GoogleGenerativeAIEmbeddings(
    model="text-embedding-005",
    vertexai=True,
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=f"{PERSIST_DIR}/{COLLECTION_NAME}/chroma",
)

retriever = RaptorRetriever(
    vectorstore=vectorstore,
    nodes=nodes,
    tree_structure=tree_structure,
    default_mode="tree_traversal",
)


/Users/katherynne/Documents/itj/slack-bot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ RAPTOR loaded | Nodes: 15 | Levels: 3


In [2]:
# Question:

question = (
    "How is the right ventricular Tei index used to predict postoperative "
    "complications in patients undergoing cardiovascular surgery?"
)

print("🧠 QUESTION:\n", question)


🧠 QUESTION:
 How is the right ventricular Tei index used to predict postoperative complications in patients undergoing cardiovascular surgery?


In [3]:
# Raptor retrieval:

raw_results = retriever.retrieve(
    query=question,
    top_k=6,
    mode="tree_traversal",
)

raw_results


[{'text': 'This text describes two separate studies:\n\n1.  **Brugada Syndrome (BS) Prediction:** A study of 221 patients found that prominent S-waves in lead II (duration ≥ 19 ms) and J-waves in V1 (amplitude ≥ 0.05 mV) on a baseline ECG significantly predicted a positive ajmaline challenge (AC) in patients suspected of BS without a spontaneous diagnostic ECG. These markers can help identify individuals with a higher likelihood of a positive AC.\n2.  **Miller Syndrome (MS) Phenotypic Expansion:** The largest cohort study to date (10 individuals from 7 families) on Miller Syndrome, caused by `DHODH` variants, expanded its known phenotypic spectrum. Novel findings included optic atrophy, frequent preaxial limb involvement (e.g., thumb/tibial hypoplasia), camptodactyly, and facial nevus, alongside previously known postaxial limb defects and congenital heart defects. The study highlights the importance of considering these new features for diagnosis and screening.',
  'score': 1.177484750

In [5]:
# Agent
from IPython.display import display, Markdown

review_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0,
    vertexai=True,
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

def review_raptor_results(question, results):
    fragments = []
    for i, r in enumerate(results):
        fragments.append(
            f"Fragment {i+1}:\n"
            f"Source: {r['source_doc']}\n"
            f"Level: {r['level']}\n"
            f"Text:\n{r['text']}\n"
        )

    prompt = f"""
You are a medical review agent.

Question:
{question}

Below are text fragments retrieved from a hierarchical retrieval system.
Your task:
- Select ONLY the fragments that directly answer the question.
- Discard fragments related to unrelated diseases, studies, or outcomes.
- Prefer original study findings over summaries.
- Return a clean, concise synthesis using ONLY the selected fragments.
- Do NOT introduce external knowledge.

Fragments:
{chr(10).join(fragments)}
"""

    response = review_llm.invoke(prompt)
    return response.content

validated_answer = review_raptor_results(question, raw_results)


display(Markdown(f"""
### ✅ Agent-Reviewed Answer

**Question**  
> {question}

**Answer**  
{validated_answer}
"""))




### ✅ Agent-Reviewed Answer

**Question**  
> How is the right ventricular Tei index used to predict postoperative complications in patients undergoing cardiovascular surgery?

**Answer**  
The right ventricular Tei index (RV-TI) is an echocardiographic measure that integrates both systolic and diastolic parameters of ventricular function. It is calculated as the ratio of the sum of isovolumetric relaxation time (IVRT) and isovolumetric contraction time (IVCT) to the ejection time (ET): TI = (IVRT + IVCT)/ET. A high Tei index, typically above 0.40 in adults, can effectively predict an increased risk of postsurgical complications, prolonged hospitalization, and other adverse events in patients undergoing cardiovascular surgery.
